# 🔍 Notebook 3: Arabic Semantic Search & Retrieval

**Project:** Deep Learning Based Arabic Audio Understanding and Retrieval System  
**Task:** Semantic Embedding + FAISS Vector Search  
**Model:** `paraphrase-multilingual-MiniLM-L12-v2` (384-dim multilingual embeddings)  
**Index:** FAISS IndexFlatIP (Inner Product = Cosine Similarity)  
**Metrics:** Precision@K, Recall@K  

**Expected Runtime:** ~5 minutes on Kaggle T4  

In [ ]:
# Cell 1: Install dependencies
!pip install -q sentence-transformers faiss-cpu tqdm
print("✅ Installation complete.")

In [ ]:
# Cell 2: Imports
import os
import pickle
import numpy as np
import pandas as pd
import faiss
import matplotlib.pyplot as plt
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm

print("All imports successful.")

## 1. Load Data

We load the `summaries.csv` from Notebook 2 (which contains transcripts + summaries).  
We will embed the **transcript text** (prediction column) for semantic search.

**Note:** Upload `summaries.csv` as a Kaggle dataset first.

In [ ]:
# Cell 3: Load data

# === UPDATE THIS PATH ===
SUMMARIES_PATH = "/kaggle/input/summaries-csv/summaries.csv"
# Alternative: use transcripts.csv directly if summaries not available
# SUMMARIES_PATH = "/kaggle/input/transcripts-csv/transcripts.csv"

if not os.path.exists(SUMMARIES_PATH):
    raise FileNotFoundError(
        f"Data not found at {SUMMARIES_PATH}. "
        "Upload summaries.csv or transcripts.csv from previous notebooks."
    )

df = pd.read_csv(SUMMARIES_PATH, encoding="utf-8-sig")
print(f"Loaded {len(df)} entries.")
print(f"Columns: {list(df.columns)}")

# Determine which column to embed
if "prediction" in df.columns:
    text_col = "prediction"
elif "reference" in df.columns:
    text_col = "reference"
else:
    text_col = df.columns[0]

print(f"Using column '{text_col}' for embedding.")

# Filter out empty/NaN texts
df = df[df[text_col].notna()].copy()
df = df[df[text_col].astype(str).str.len() > 3].copy()
df = df.reset_index(drop=True)
print(f"Valid entries for indexing: {len(df)}")

# Preview
display(df[["id", text_col]].head())

## 2. Load Embedding Model

We use **`paraphrase-multilingual-MiniLM-L12-v2`** — the same model used in the project's `embedder.py`.  
This model produces 384-dimensional normalized embeddings that support Arabic, English, and 50+ languages.

In [ ]:
# Cell 4: Load embedding model (mirrors friend's embedder.py)
EMBED_MODEL_NAME = "paraphrase-multilingual-MiniLM-L12-v2"

print(f"Loading embedding model: {EMBED_MODEL_NAME} ...")
embed_model = SentenceTransformer(EMBED_MODEL_NAME)
print(f"✅ Loaded. Embedding dimension: {embed_model.get_sentence_embedding_dimension()}")

In [ ]:
# Cell 5: Create embeddings for all transcripts (mirrors friend's embedder.py)

texts = df[text_col].astype(str).tolist()

print(f"Embedding {len(texts)} texts...")
embeddings = embed_model.encode(
    texts,
    convert_to_numpy=True,
    normalize_embeddings=True,  # Normalize for cosine similarity via inner product
    batch_size=32,
    show_progress_bar=True
).astype(np.float32)

print(f"✅ Embeddings shape: {embeddings.shape}")
print(f"   Each vector: {embeddings.shape[1]} dimensions")

## 3. Build FAISS Index

We build a **FAISS IndexFlatIP** (Inner Product) index — the same type used in the project's `search.py`.  
Since embeddings are L2-normalized, inner product equals cosine similarity.

In [ ]:
# Cell 6: Build FAISS index (mirrors friend's search.py)

dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)  # Inner product = cosine similarity
index.add(embeddings)

print(f"✅ FAISS index built: {index.ntotal} vectors indexed")
print(f"   Index type: IndexFlatIP (cosine similarity)")

# Save index and metadata
faiss.write_index(index, "/kaggle/working/corpus.index")

metadata_list = []
for idx, row in df.iterrows():
    metadata_list.append({
        "filename": str(row.get("id", f"sample_{idx}")),
        "transcript": str(row[text_col]),
        "summary": str(row.get("summary", ""))
    })

with open("/kaggle/working/metadata.pkl", "wb") as f:
    pickle.dump(metadata_list, f)

print(f"💾 Saved: corpus.index + metadata.pkl")

In [ ]:
# Cell 7: Search function (mirrors friend's search.py)

def search_index(query_text, top_k=5):
    """Search the FAISS index with a text query."""
    # Embed the query
    query_vec = embed_model.encode(
        [query_text],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype(np.float32)
    
    # Search
    scores, indices = index.search(query_vec, top_k)
    
    results = []
    for score, idx in zip(scores[0], indices[0]):
        if 0 <= idx < len(metadata_list):
            entry = metadata_list[idx].copy()
            entry["similarity_score"] = float(score)
            entry["index"] = int(idx)
            results.append(entry)
    
    return results

print("✅ Search function ready.")

## 4. Run Search Queries

We test 5 Arabic queries across different topics to demonstrate the semantic search capability.

In [ ]:
# Cell 8: Run search queries
DEFAULT_TOP_K = 5

test_queries = [
    "الحب والعلاقات",          # Love and relationships
    "الطبيعة والبيئة",         # Nature and environment
    "الدين والإيمان",          # Religion and faith
    "الصبر والحكمة",           # Patience and wisdom
    "التعليم والمدارس",        # Education and schools
]

all_search_results = {}

for query in test_queries:
    print("\n" + "=" * 70)
    print(f"🔍 Query: {query}")
    print("=" * 70)
    
    results = search_index(query, top_k=DEFAULT_TOP_K)
    all_search_results[query] = results
    
    for rank, result in enumerate(results, 1):
        score_pct = min(result['similarity_score'] * 100, 100)
        print(f"\n  #{rank} | Score: {result['similarity_score']:.4f} ({score_pct:.1f}%)")
        print(f"       File: {result['filename']}")
        print(f"       Text: {result['transcript'][:150]}")

## 5. Search Evaluation: Precision@K & Recall@K

We evaluate search quality using **Precision@K** and **Recall@K** — the same metrics from the project's `evaluator.py`.

Since we don't have ground-truth relevance labels, we use a **threshold-based approach**: results with similarity > 0.4 are considered "relevant".

In [ ]:
# Cell 9: Precision@K and Recall@K (mirrors friend's evaluator.py)

def compute_precision_at_k(retrieved_ids, relevant_ids, k):
    """Fraction of top-K results that are relevant."""
    retrieved_top_k = retrieved_ids[:k]
    relevant_set = set(relevant_ids)
    hits = sum(1 for item in retrieved_top_k if item in relevant_set)
    return round(hits / k, 4) if k > 0 else 0.0

def compute_recall_at_k(retrieved_ids, relevant_ids, k):
    """Fraction of relevant items found in top-K."""
    retrieved_top_k = retrieved_ids[:k]
    relevant_set = set(relevant_ids)
    if not relevant_set:
        return 0.0
    hits = sum(1 for item in retrieved_top_k if item in relevant_set)
    return round(hits / len(relevant_set), 4)

# Use threshold-based relevance: similarity > 0.4 = relevant
RELEVANCE_THRESHOLD = 0.4
K = DEFAULT_TOP_K

eval_rows = []

for query, results in all_search_results.items():
    # Get all IDs with score > threshold from a broader search
    broad_results = search_index(query, top_k=20)
    relevant_ids = [r["index"] for r in broad_results if r["similarity_score"] > RELEVANCE_THRESHOLD]
    retrieved_ids = [r["index"] for r in results]
    
    p_at_k = compute_precision_at_k(retrieved_ids, relevant_ids, K)
    r_at_k = compute_recall_at_k(retrieved_ids, relevant_ids, K)
    
    avg_score = np.mean([r["similarity_score"] for r in results]) if results else 0
    
    eval_rows.append({
        "query": query,
        f"precision@{K}": p_at_k,
        f"recall@{K}": r_at_k,
        "avg_similarity": round(avg_score, 4),
        "relevant_found": len(relevant_ids)
    })

df_search_eval = pd.DataFrame(eval_rows)

print("=" * 60)
print(f"SEARCH EVALUATION — Precision@{K} & Recall@{K}")
print("=" * 60)
display(df_search_eval)

mean_precision = df_search_eval[f"precision@{K}"].mean()
mean_recall = df_search_eval[f"recall@{K}"].mean()
print(f"\nMean Precision@{K}: {mean_precision:.4f}")
print(f"Mean Recall@{K}:    {mean_recall:.4f}")

In [ ]:
# Cell 10: Save all results

# Save search evaluation
df_search_eval.to_csv("/kaggle/working/search_eval.csv", index=False, encoding="utf-8-sig")

# Save detailed search results
search_detail_rows = []
for query, results in all_search_results.items():
    for rank, r in enumerate(results, 1):
        search_detail_rows.append({
            "query": query,
            "rank": rank,
            "filename": r["filename"],
            "similarity_score": r["similarity_score"],
            "transcript": r["transcript"][:200]
        })

pd.DataFrame(search_detail_rows).to_csv(
    "/kaggle/working/search_results.csv", index=False, encoding="utf-8-sig"
)

print("💾 Saved: search_eval.csv, search_results.csv")

print("\n" + "=" * 60)
print("📋 RESULTS TABLE FOR REPORT")
print("=" * 60)
print("| Metric | Value |")
print("|--------|-------|")
print(f"| Embedding Model | {EMBED_MODEL_NAME} |")
print(f"| Index Type | FAISS IndexFlatIP |")
print(f"| Vectors Indexed | {index.ntotal} |")
print(f"| Embedding Dim | {dimension} |")
print(f"| Mean Precision@{K} | {mean_precision:.4f} |")
print(f"| Mean Recall@{K} | {mean_recall:.4f} |")
print("=" * 60)

In [ ]:
# Cell 11: Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Average similarity per query
ax1 = axes[0]
query_labels = [q[:20] + "..." if len(q) > 20 else q for q in df_search_eval["query"]]
ax1.barh(query_labels, df_search_eval["avg_similarity"], 
         color=["#3498db", "#2ecc71", "#9b59b6", "#e74c3c", "#f39c12"],
         edgecolor="black", alpha=0.85)
ax1.set_xlabel("Average Cosine Similarity", fontsize=11)
ax1.set_title("Search Relevance per Query", fontsize=13, fontweight="bold")
ax1.grid(axis="x", alpha=0.3)
for i, v in enumerate(df_search_eval["avg_similarity"]):
    ax1.text(v + 0.005, i, f"{v:.3f}", va="center", fontsize=10)

# Plot 2: Precision@K and Recall@K
ax2 = axes[1]
x = np.arange(len(query_labels))
width = 0.35
bars1 = ax2.bar(x - width/2, df_search_eval[f"precision@{K}"], width,
                label=f"Precision@{K}", color="#ffb86c", edgecolor="black")
bars2 = ax2.bar(x + width/2, df_search_eval[f"recall@{K}"], width,
                label=f"Recall@{K}", color="#8be9fd", edgecolor="black")
ax2.set_ylabel("Score", fontsize=11)
ax2.set_title(f"Precision@{K} & Recall@{K}", fontsize=13, fontweight="bold")
ax2.set_xticks(x)
ax2.set_xticklabels([q[:12] for q in df_search_eval["query"]], fontsize=9)
ax2.legend()
ax2.set_ylim(0, 1.1)
ax2.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig("/kaggle/working/search_evaluation.png", dpi=200, bbox_inches="tight")
plt.show()
print("💾 Plot saved: /kaggle/working/search_evaluation.png")

## Related Work

1. **Sentence-BERT:** Reimers, N. and Gurevych, I. (2019). "Sentence-BERT: Sentence Embeddings using Siamese BERT-Networks." *EMNLP 2019.* The foundation for multilingual sentence embeddings used in this search system.

2. **FAISS:** Johnson, J., Douze, M., and Jégou, H. (2019). "Billion-scale similarity search with GPUs." *IEEE Transactions on Big Data.* The vector search library used for efficient nearest neighbor retrieval.

3. **ARCD:** Mozannar, H., et al. (2019). "Neural Arabic Question Answering." *WANLP 2019.* Arabic reading comprehension benchmark for evaluating search and retrieval systems.

### Known Limitations
- **Short texts:** Very short transcripts (1-3 words) have less semantic signal for meaningful search.
- **MiniLM size:** The 384-dim model is fast but less precise than larger models (768-dim).
- **No reranking:** Results are based solely on embedding similarity without cross-encoder reranking.

## Results Summary

| Item | Value |
|------|-------|
| Embedding Model | paraphrase-multilingual-MiniLM-L12-v2 |
| Index Type | FAISS IndexFlatIP |
| Vectors Indexed | *(fill after running)* |
| Mean Precision@K | *(fill after running)* |
| Mean Recall@K | *(fill after running)* |
| GPU Used | Tesla T4 |